## Summarise all the *kelch13* mutations that were observed

Note:
- Only WT if it is WT at that codon position across all

In [1]:
import os
import pandas as pd
import geopandas as gpd

from statsmodels.stats.proportion import proportion_confint

## Settings

In [3]:
DIR_SEQDATA = "../seqdata/all/summaries/HRP23_MIS2024"
DIR_GEODATA = "../geodata/"
save_results = False

## Functions

In [4]:
VALIDATED = [
    "F446I", "N458Y", "C469Y", "M476I", "Y493H",
    "G533S", "R539T", "I543T", "P553L", "R561H",
    "P574L", "C580Y", "R622I", "A675V"
]
CANDIDATE = [
    "E252Q", "P441L", "C469F", "N537I", "G538V", "V568G"
]
def annotate_kelch13_mutation(mut: str) -> str:
    if mut in VALIDATED:
        return f"{mut}$^a$"
    if mut in CANDIDATE:
        return f"{mut}$^b$"
    return mut

## Load National data

In [5]:
df_national = (
    pd.read_csv(
        f"{DIR_SEQDATA}/summary.variants.prevalence.csv"
    )
    .query("gene == 'kelch13'")
    #.drop(["gene", "chrom", "mut_type", "aa_pos", "aa_change"], axis=1)
)

In [6]:
# Rounding of prevalence to one decimal place, avoid arbitrary precision and clarity in paper
df_national["prevalence"] = df_national["prevalence"].round(1)

In [7]:
# Annotations
df_national['n_total'] = df_national['n_mut'] + df_national['n_mixed']
df_national['summary'] = [
    f"{r['prevalence']:.1f}% ({int(r['n_total'])}/{int(r['n_passed'])})"
    for _, r in df_national.iterrows()
]
df_national["prevalence_table"] = [
    f"{r['prevalence']:.1f} ({r['prevalence_lowci']:.1f}-{r['prevalence_highci']:.1f})"
    for _, r in df_national.iterrows()
]
df_national["aa_change_who"] = [annotate_kelch13_mutation(m) for m in df_national["aa_change"]]

**Formatting**

In [8]:
column_names = {
    "aa_change_who": "Mutation",
    "prevalence_table": "Prevalence (95% CI) $-$ %",
    "n_passed": "N",
    "n_mut": "Mutant $-$ n",
    "n_mixed": "Mixed $-$ n",
    "n_total": "Mutant or Mixed $-$ n"
}

**All mutations**

In [9]:
df_kelch13 = df_national[column_names.keys()].rename(column_names, axis=1)

In [10]:
df_kelch13.head() # need to clean this

,Mutation,Prevalence (95% CI) $-$ %,N,Mutant $-$ n,Mixed $-$ n,Mutant or Mixed $-$ n
85,D389E,0.3 (0.1-0.7),1934,0,6,6
86,S400I,0.1 (0.0-0.3),1948,0,1,1
87,R411K,0.3 (0.1-0.7),1946,0,6,6
88,R411E,0.1 (0.0-0.4),1946,0,2,2
89,P419S,0.1 (0.0-0.3),1949,1,0,1


**Prepare main table**

In [11]:
LOW_PREV = 0.5

In [12]:
df_kelch13_main = df_national.query("prevalence >= @LOW_PREV")[column_names.keys()].rename(column_names, axis=1)
df_kelch13_main

,Mutation,Prevalence (95% CI) $-$ %,N,Mutant $-$ n,Mixed $-$ n,Mutant or Mixed $-$ n
92,P441L$^b$,2.6 (1.9-3.4),1948,28,22,50
95,C473F,0.6 (0.3-1.0),1950,4,7,11
100,G533A,0.8 (0.4-1.3),1950,7,8,15
109,P574L$^a$,0.5 (0.2-0.9),1950,2,7,9
110,A578S,1.0 (0.6-1.5),1950,6,13,19
114,R622T,0.8 (0.5-1.3),1950,8,8,16
121,P667A,1.5 (1.0-2.2),1949,10,20,30
126,A724E,11.7 (10.4-13.3),1949,111,118,229


**Add low frequency mutations**

In [13]:
df_kelch13_low = df_national.query("prevalence < @LOW_PREV")

In [14]:
def compute_over_arbitrary_set(df_set: pd.DataFrame, set_name: str) -> pd.DataFrame:
    N = df_set.n_passed.max() # a bit dubious, but only a bit
    n = df_set.n_total.sum()
    low, high = proportion_confint(n, N, alpha=0.5, method="beta")
    
    df_kelch13_other = pd.DataFrame(
        {"aa_change_who": set_name,
         "prevalence_table": f"{100*n/N:.1f} ({100*low:.1f}-{100*high:.1f})",
         "n_passed": df_set.n_passed.max(),
         "n_mut": df_set.n_mut.sum(),
         "n_mixed": df_set.n_mixed.sum(),
         "n_total": df_set.n_total.sum()
        }, index=[0]
    ).rename(column_names, axis=1)

    return df_kelch13_other

In [15]:
df_kelch13_other = compute_over_arbitrary_set(df_national.query("prevalence < @LOW_PREV"), set_name="Other$^c$")
df_kelch13_all = compute_over_arbitrary_set(df_national, set_name="Total")
#df_kelch13_all_but_a724e = compute_over_arbitrary_set(df_national.query("aa_change_who != 'A724E'"), set_name="All $-$ A724E")

In [16]:
df_kelch13_combined = pd.concat([
    df_kelch13_main, 
    df_kelch13_other,
    df_kelch13_all,
    #df_kelch13_all_but_a724e
], axis=0).reset_index(drop=True)

In [17]:
df_kelch13_combined

,Mutation,Prevalence (95% CI) $-$ %,N,Mutant $-$ n,Mixed $-$ n,Mutant or Mixed $-$ n
0,P441L$^b$,2.6 (1.9-3.4),1948,28,22,50
1,C473F,0.6 (0.3-1.0),1950,4,7,11
2,G533A,0.8 (0.4-1.3),1950,7,8,15
3,P574L$^a$,0.5 (0.2-0.9),1950,2,7,9
4,A578S,1.0 (0.6-1.5),1950,6,13,19
5,R622T,0.8 (0.5-1.3),1950,8,8,16
6,P667A,1.5 (1.0-2.2),1949,10,20,30
7,A724E,11.7 (10.4-13.3),1949,111,118,229
8,Other$^c$,4.3 (4.0-4.7),1950,19,65,84
9,Total,23.7 (23.1-24.4),1950,195,268,463


## Write

In [18]:
if save_results:
    #df_kelch13_main.to_excel(f"../tables/stable_kelch13.xlsx", sheet_name="Table 1", index=False)
    #df_kelch13_other.to_excel(f"../tables/table1_kelch13-other.xlsx", sheet_name="Table 1", index=False)
    df_kelch13_combined.to_excel(f"../tables/table_kelch13-other-total.xlsx", sheet_name="Table 1", index=False)
    df_kelch13.to_excel(f"../tables/stable_kelch13-all.xlsx", sheet_name="STable 1", index=False)

## Text for paper

In [19]:
print(f"""
    SUMMARY
    --------------------------------------------------------------------------------------------------------------------
    A total of {df_national.shape[0]} unique kelch13 mutations were observed across {df_national.n_total.sum()} samples.
    Of these, {df_national.query("prevalence >= @LOW_PREV").shape[0]} mutations were at >={LOW_PREV} across {df_national.query("prevalence > @LOW_PREV")['n_total'].sum()} samples.
    And {df_national.query("prevalence < @LOW_PREV").shape[0]} were below.
    
""")


    SUMMARY
    --------------------------------------------------------------------------------------------------------------------
    A total of 42 unique kelch13 mutations were observed across 463 samples.
    Of these, 8 mutations were at >=0.5 across 370 samples.
    And 34 were below.




In [20]:
42 - 8

34

In [21]:
# make it easier to grab columns
df_kelch13 = df_kelch13.rename({v: k for k, v in column_names.items()}, axis=1)

In [22]:
df_kelch13.sort_values("n_total", ascending=False).head() # most common mutations

,aa_change_who,prevalence_table,n_passed,n_mut,n_mixed,n_total
126,A724E,11.7 (10.4-13.3),1949,111,118,229
92,P441L$^b$,2.6 (1.9-3.4),1948,28,22,50
121,P667A,1.5 (1.0-2.2),1949,10,20,30
110,A578S,1.0 (0.6-1.5),1950,6,13,19
114,R622T,0.8 (0.5-1.3),1950,8,8,16


- Only 3 mutations at >1% prevalence

In [23]:
df_national.query("aa_change in @VALIDATED")[column_names.keys()].rename(column_names,axis=1)

,Mutation,Prevalence (95% CI) $-$ %,N,Mutant $-$ n,Mixed $-$ n,Mutant or Mixed $-$ n
109,P574L$^a$,0.5 (0.2-0.9),1950,2,7,9
122,A675V$^a$,0.3 (0.1-0.6),1950,2,3,5


In [24]:
df_national.query("aa_change in @CANDIDATE")[column_names.keys()].rename(column_names,axis=1)

,Mutation,Prevalence (95% CI) $-$ %,N,Mutant $-$ n,Mixed $-$ n,Mutant or Mixed $-$ n
92,P441L$^b$,2.6 (1.9-3.4),1948,28,22,50
